# Kaggle offline anchor — ENVEDA CASMI v2-A7

Attach two inputs only: (1) the private Kaggle Model `enveda-casmi-v2-a7/sklearn/full-e2e`, and (2) the competition input containing `test.parquet` and `sample_submission.csv`. Internet stays OFF. The notebook resolves everything by filenames under `/kaggle/input` and runs the same `FILENAMES_ONLY` path validated in Colab.


In [ ]:
# 1. Resolve the attached Kaggle Model runtime source by filename
from pathlib import Path
import json, os, sys

INPUT_ROOT = Path('/kaggle/input').resolve()
WORK_DIR = Path('/kaggle/working').resolve()
DEVICE_PREFERENCE = 'gpu'
REQUIRE_GPU = False

runtime_hits = sorted(INPUT_ROOT.rglob('runtime/src/casmi_runtime/named_inference.py'))
if len(runtime_hits) != 1:
    raise RuntimeError(f'Expected exactly one attached ENVEDA runtime model, found {len(runtime_hits)}: {runtime_hits}')
MODEL_PAYLOAD_ROOT = runtime_hits[0].parents[3]
RUNTIME_SRC = MODEL_PAYLOAD_ROOT / 'runtime' / 'src'
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))

print('MODEL_PAYLOAD_ROOT:', MODEL_PAYLOAD_ROOT)
print('RUNTIME_SRC       :', RUNTIME_SRC)


In [ ]:
# 2. Resolve bundle + competition files by filename only
bundle_hits = sorted(INPUT_ROOT.rglob('bundle/config.json'))
bundle_hits = [p.parent for p in bundle_hits if (p.parent / 'code' / 'casmi_infer' / '__init__.py').exists()]
if len(bundle_hits) != 1:
    raise RuntimeError(f'Expected exactly one ENVEDA bundle root, found {len(bundle_hits)}: {bundle_hits}')
BUNDLE_ROOT = bundle_hits[0]

test_hits = [p for p in INPUT_ROOT.rglob('test.parquet') if BUNDLE_ROOT not in p.parents]
sample_hits = [p for p in INPUT_ROOT.rglob('sample_submission.csv') if BUNDLE_ROOT not in p.parents]
if len(test_hits) != 1:
    raise RuntimeError(f'Expected exactly one test.parquet outside the model bundle, found {test_hits}')
if len(sample_hits) != 1:
    raise RuntimeError(f'Expected exactly one sample_submission.csv outside the model bundle, found {sample_hits}')
TEST_PATH = test_hits[0]
SAMPLE_SUBMISSION_PATH = sample_hits[0]

print('BUNDLE_ROOT       :', BUNDLE_ROOT)
print('TEST_PATH         :', TEST_PATH)
print('SAMPLE_SUBMISSION :', SAMPLE_SUBMISSION_PATH)
print('WORK_DIR          :', WORK_DIR)


In [ ]:
# 3. Environment / accelerator diagnostics — no installs, no network
from importlib.metadata import PackageNotFoundError, version
from casmi_runtime.accelerator import detect_accelerator

def _v(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

ACCEL = detect_accelerator()
ENV = {k: _v(k) for k in ('numpy','pandas','pyarrow','lightgbm','numba','psutil','rdkit')}
print(json.dumps({'accelerator': ACCEL, 'packages': ENV}, indent=2, default=str))
print('Internet expected: OFF')
print('Network required : False')


In [ ]:
# 4. Frozen identity checks from named files
from casmi_runtime.config import RuntimeConfig
from casmi_runtime.named_inference import run_named_inference

CONFIG = json.loads((BUNDLE_ROOT / 'config.json').read_text(encoding='utf-8'))
MODEL_INFO = json.loads((BUNDLE_ROOT / 'models' / 'model_info.json').read_text(encoding='utf-8'))
aggregator_name = CONFIG.get('aggregator_name') or (CONFIG.get('aggregator') or {}).get('name')

assert CONFIG.get('bundle_version') == 'v2-A7', CONFIG.get('bundle_version')
assert CONFIG.get('CONFIG_HASH') == '60174e39a2a3c6b4', CONFIG.get('CONFIG_HASH')
assert CONFIG.get('model_id') == 'V1_TL_1K_TESTSIM_STRICT', CONFIG.get('model_id')
assert MODEL_INFO.get('model_id') == 'V1_TL_1K_TESTSIM_STRICT', MODEL_INFO.get('model_id')
assert MODEL_INFO.get('freeze_status') == 'FROZEN', MODEL_INFO.get('freeze_status')
assert aggregator_name == 'MOST_CONFIDENT_SPECTRUM', aggregator_name
assert float(CONFIG.get('calibration_temperature')) == 1.1947045372735254

print('bundle_version:', CONFIG['bundle_version'])
print('CONFIG_HASH   :', CONFIG['CONFIG_HASH'])
print('model_id      :', CONFIG['model_id'])
print('aggregator    :', aggregator_name)
print('temperature   :', CONFIG['calibration_temperature'])


In [ ]:
# 5. RuntimeConfig for explicit Kaggle Model + competition filenames
CFG = RuntimeConfig(
    runtime='kaggle',
    input_root=INPUT_ROOT,
    work_dir=WORK_DIR,
    device_preference=DEVICE_PREFERENCE,
    require_gpu=REQUIRE_GPU,
    expected_config_hash='60174e39a2a3c6b4',
    extra={
        'input_mode': 'FILENAMES_ONLY',
        'kaggle_model_payload_root': str(MODEL_PAYLOAD_ROOT),
        'bundle_root': str(BUNDLE_ROOT),
        'test_file': TEST_PATH.name,
        'sample_submission_file': SAMPLE_SUBMISSION_PATH.name,
        'network_required': False,
        'internet_expected': 'OFF',
        'kaggle_kernel_run_type': os.environ.get('KAGGLE_KERNEL_RUN_TYPE'),
    },
)
print(json.dumps(CFG.as_dict(), indent=2, default=str))


In [ ]:
# 6. Full E2E functional gate + inference
REPORT = run_named_inference(
    CFG,
    bundle_dir=BUNDLE_ROOT,
    test_path=TEST_PATH,
    sample_path=SAMPLE_SUBMISSION_PATH,
    allow_feature_only_drift=True,
)


In [ ]:
# 7. Deterministic Colab-vs-Kaggle anchor checks
EXPECTED = {
    'CONFIG_HASH': '60174e39a2a3c6b4',
    'model_id': 'V1_TL_1K_TESTSIM_STRICT',
    'aggregator': 'MOST_CONFIDENT_SPECTRUM',
    'calibration_temperature': 1.1947045372735254,
    'n_spectra': 1213,
    'n_molecules': 400,
    'candidate_pair_count': 471173,
    'similarity_evaluations': 2057028,
    'exception_count': 0,
    'unsupported_adducts': [],
    'submission_rows': 400,
}

mismatches = {}
for key, expected in EXPECTED.items():
    actual = REPORT.get(key)
    if actual != expected:
        mismatches[key] = {'expected': expected, 'actual': actual}

hard_selftest = [
    'candidate_set_mismatches', 'score_mismatches', 'fold_score_mismatches',
    'rank_mismatches', 'prob_mismatches', 'selected_spectrum_mismatches',
    'confidence_mismatches', 'molecule_ranking_mismatches',
]
for key in hard_selftest:
    actual = (REPORT.get('self_test_mismatches') or {}).get(key)
    if actual != 0:
        mismatches[f'self_test.{key}'] = {'expected': 0, 'actual': actual}

if REPORT.get('self_test_status') != 'PASS':
    mismatches['self_test_status'] = {'expected': 'PASS', 'actual': REPORT.get('self_test_status')}
if REPORT.get('self_test_downstream_exact') is not True:
    mismatches['self_test_downstream_exact'] = {'expected': True, 'actual': REPORT.get('self_test_downstream_exact')}
if REPORT.get('submission_validation_status') != 'PASS':
    mismatches['submission_validation_status'] = {'expected': 'PASS', 'actual': REPORT.get('submission_validation_status')}
if REPORT.get('gpu_used') is not False:
    mismatches['gpu_used'] = {'expected': False, 'actual': REPORT.get('gpu_used')}

ANCHOR = {
    'expected': EXPECTED,
    'actual': {k: REPORT.get(k) for k in EXPECTED},
    'feature_mismatches': REPORT.get('self_test_feature_mismatches'),
    'strict_self_test_passed': REPORT.get('self_test_strict_passed'),
    'downstream_exact': REPORT.get('self_test_downstream_exact'),
    'mismatches': mismatches,
    'anchor_status': 'PASS' if not mismatches else 'FAIL',
}
(WORK_DIR / 'kaggle_anchor_report.json').write_text(json.dumps(ANCHOR, indent=2, default=str), encoding='utf-8')
print(json.dumps(ANCHOR, indent=2, default=str))
if mismatches:
    raise RuntimeError(f'Kaggle anchor FAILED: {mismatches}')


In [ ]:
# 8. Final inspection
import pandas as pd

assert REPORT['bundle_integrity_mode'] == 'FILENAMES_ONLY'
assert REPORT['bundle_hash_verification'] is False
assert REPORT['fixture_hash_verification'] is False
assert Path(REPORT['casmi_infer_source']).resolve().is_relative_to((BUNDLE_ROOT / 'code').resolve())
assert (WORK_DIR / 'submission.csv').exists()
assert (WORK_DIR / 'run_report.json').exists()
assert (WORK_DIR / 'kaggle_anchor_report.json').exists()

submission = pd.read_csv(WORK_DIR / 'submission.csv')
print('submission shape:', submission.shape)
print('backend         :', REPORT.get('actual_backend'))
print('fallback_reason :', REPORT.get('fallback_reason'))
print('feature drift   :', REPORT.get('self_test_feature_mismatches'))
print('downstream exact:', REPORT.get('self_test_downstream_exact'))
print('outputs         :', WORK_DIR / 'submission.csv', WORK_DIR / 'run_report.json', WORK_DIR / 'kaggle_anchor_report.json')
print('KAGGLE OFFLINE ANCHOR PASS')
